##<font color=blue>**Problem Statement**

### Business Context

NewsFindr is redefining news discovery by delivering real-time news updates tailored to user interests. Traditional search methods and generic news feeds often lead to information overload and inefficiencies, making it challenging for users to access relevant and trustworthy content efficiently.

To address this, NewsFindr wants to leverage Agentic AI to build an AI-powered news retrieval agent that ensures accuracy and credibility. By utilizing a structured, multi-step approach, the system will provide secure, fair, and explainable recommendations - enhancing user engagement, optimizing content discovery, and improving access to timely and relevant news.

### Objective

- Provide real-time, personalized news retrieval to help users discover relevant
content effortlessly.

- Ensure accuracy and credibility by sourcing news from trusted platforms and minimizing misinformation.

- Improve user engagement through seamless content discovery, reducing information overload.

- Streamline the news consumption process by eliminating outdated and irrelevant content, providing a refined reading experience.

##<font color=blue>**Please read the instructions carefully before starting the project**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.
* Blanks '_____' are provided in the notebook that
needs to be filled with an appropriate code to get the correct result. With every '_____' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
* Add the results/observations at the end of the analysis and submit the same.

##<font color=blue>**Installing and Importing Necessary Libraries and Dependencies**

In [ ]:
# Single, pinned install cell. Do NOT run any other pip install later in this notebook

%pip install -qU \
    langchain==0.3.27 \
    langchain-core==0.3.75 \
    langchain-community==0.3.27 \
    langchain-groq==0.3.8 \
    langgraph==0.6.6 \
    ddgs==9.6.1 \
    numpy>=2.1.0      # Python < 3.13 requires NumPy 2.0.2 to resolve dependency issues.
                      # Google Colab users can skip this (Colab already uses Python 3.13+)

**Note:**

- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

###**Imports**

In [ ]:
# Core Python standard utilities for system tasks, parsing, timing, and type annotations
import json
import os
import re
import time
import random
import sqlite3
from typing import List, Dict, Any, Optional

# Data analysis and manipulation library
import pandas as pd

# LangChain message abstractions for structuring agent prompts
from langchain_core.messages import SystemMessage, HumanMessage

# Custom tool wrappers for binding functions to LangChain agents
from langchain_core.tools import Tool, StructuredTool
from langchain_groq import ChatGroq

# LangChain database wrappers and automated SQL agent toolkits
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langgraph.prebuilt import create_react_agent

# Pydantic schema validation for strict tool parameter definitions
from pydantic import BaseModel, Field

# DuckDuckGo web search client for real-time web data extraction
from ddgs import DDGS

# Suppress non-critical deprecation warnings for cleaner notebook output
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

#<font color=blue>**Utilities**

####**Note**

This section from **Token-budget settings** to **Safe LLM call helper** focuses on demonstrating how to apply limits on LLM calls and invocation requests.

- The primary goal here is to show how we can **protect the LLM from being overused or exhausted** by putting sensible controls in place.

- You are *not* required to go through every line of code or deeply understand the full execution flow to benefit from this section. The code is provided mainly as a **safety and performance measure**, not as core learning content.

- Treat this as a reference for good practices in managing LLM usage, rather than a must‑understand code block.

###**Token-budget settings**

**Configuration & Rate Limit Controls**

This block defines central rules for API traffic, text processing limits, and AI execution to keep the application stable, fast, and cost-effective on free-tier services.

| Setting Category | Purpose | Key Variables |
| :--- | :--- | :--- |
| **Rate Limit Protection** | Prevents API blocks by capping tokens sent per minute and auto-retrying requests if rate limits (Error 429) are hit. | `TPM_LIMIT`, `TPM_SAFETY`, `RATE_RETRIES` |
| **Token Budgeting** | Trims search results, body text, and database items so AI prompts stay concise, fast, and within memory caps. | `MAX_INTERESTS`, `MAX_BODY_CHARS`, `LLM_MAX_TOKENS` |
| **Workflow Control** | Caps reasoning steps to stop infinite loops and toggles between a flexible AI agent or a lower-cost fixed pipeline. | `USE_AGENT`, `AGENT_RECURSION_LIMIT` |

In [ ]:
# ---- Groq free-tier rate limits --------------------------------------------
# Base limit for Groq free-tier TPM threshold (adjust per account settings)
TPM_LIMIT      = 8000    # Maximum tokens allowed per minute
TPM_SAFETY     = 0.80    # Safety factor (80% allocation) to account for estimation variance
RATE_RETRIES   = 6       # Maximum backoff attempts when receiving HTTP 429 rate limit errors

# ---- Token budget controls -------------------------------------------------
MAX_INTERESTS          = 2     # Upper bound on user interests queried from SQLite database
MAX_RESULTS_PER_QUERY  = 3     # Maximum search snippet results kept per DuckDuckGo query
MAX_BODY_CHARS         = 1000  # Character cutoff limit applied to each web search result snippet
MAX_RESULTS_TO_FILTER  = 8     # Maximum search results passed to downstream credibility filters
MAX_URLS_TO_SUMMARIZE  = 4     # Upper limit on source URLs passed to summarization step
AGENT_RECURSION_LIMIT  = 12    # Maximum step count allowed before terminating a ReAct agent loop
LLM_MAX_TOKENS         = 2000  # Token limit capping model response generation length

# Strategy Switch: True uses autonomous ReAct agent; False uses fixed, low-cost pipeline
USE_AGENT = True

###**Rate-limit budget**

This script acts like a smart speed regulator for sending messages to the AI. It keeps track of how much text is being sent, prevents sending too much at once, and automatically pauses the program when speed limit warnings occur.

* Usage Tracker: Keeps a running 1-minute log of text sent. If sending a new message will exceed the safety allowance, it pauses the program until the time window resets.
* Length Estimator: Calculates the total word count of incoming and outgoing text before sending it, ensuring requests stay within safe boundaries.
* Error Reader: Detects "slow down" warning messages from the service, reads how many seconds to wait, and pauses safely instead of breaking the app.
* System Reset: Provides a full 60-second pause to completely clear the tracker and start fresh with a full usage allowance.

In [ ]:
class TokenBudget:
    """Rolling 60-second token budget manager."""

    def __init__(self, tpm_limit: int, safety: float = 0.8):
        self.limit = int(tpm_limit * safety)      # Usable token cap after safety margin
        self.events: List[List[float]] = []        # Tracks execution events as [timestamp, tokens]

    def _prune(self) -> None:
        """Remove usage events older than 60 seconds from the rolling window."""
        cutoff = time.time() - 60
        self.events = [e for e in self.events if e[0] > cutoff]

    def used(self) -> int:
        """Calculate total tokens consumed within the active 60-second window."""
        self._prune()
        return int(sum(e[1] for e in self.events))

    def reserve(self, tokens: int) -> None:
        """Block execution until requested tokens fit into the rolling window."""
        tokens = min(tokens, self.limit)          # Cap reservation request to maximum window limit
        while True:
            self._prune()
            # Proceed if room is available or no prior event history exists
            if self.used() + tokens <= self.limit or not self.events:
                break
            # Calculate sleep duration until the oldest logged event expires
            wait = max(61 - (time.time() - self.events[0][0]), 1.0)
            print(f"  [budget] {self.used()}/{self.limit} tokens used this minute - "
                  f"waiting {wait:.0f}s for the window to refill")
            time.sleep(wait)
        # Record new token allocation event
        self.events.append([time.time(), tokens])

    def settle(self, estimated: int, actual: int) -> None:
        """Reconcile estimated token usage with exact usage returned by API."""
        if self.events and actual > 0:
            self.events[-1][1] = min(actual, self.limit)

    def penalise(self) -> None:
        """Pessimistically mark the budget window as full upon encountering a 429 error."""
        self.events.append([time.time(), self.limit])


# Initialize global rate-limiting budget tracker instance
BUDGET = TokenBudget(TPM_LIMIT, TPM_SAFETY)


def estimate_tokens(text: Any) -> int:
    """Estimate token count using a conservative ~3 characters per token ratio."""
    return max(1, len(str(text)) // 3)


def estimate_messages_tokens(messages) -> int:
    """Sum estimated tokens across message contents, formatting overhead, and tool calls."""
    total = 0
    for m in messages:
        content = getattr(m, "content", m)        # Extract content from message object or dict
        total += estimate_tokens(content) + 4      # Include 4-token structural overhead per message
        for tc in (getattr(m, "tool_calls", None) or []):
            total += estimate_tokens(tc)          # Add token costs for attached tool call payloads
    return total


def parse_retry_after(error_text: str, default: float = 20.0) -> float:
    """Extract recommended backoff duration in seconds from API rate limit error strings."""
    # Pattern match standard seconds format (e.g., "3.53s" or "500ms")
    m = re.search(r"try again in ([\d.]+)\s*m?s", error_text, re.I)
    if m:
        secs = float(m.group(1))
        if "ms" in error_text[m.start():m.end() + 3].lower():
            secs /= 1000.0                        # Convert milliseconds to seconds
        return secs + 2.0                         # Add safety buffer to recommended wait

    # Pattern match combined minutes and seconds format (e.g., "1m30.5s")
    m = re.search(r"try again in (\d+)m([\d.]+)s", error_text, re.I)
    if m:
        return float(m.group(1)) * 60 + float(m.group(2)) + 2.0

    return default                                # Return fallback wait duration if parsing fails


def is_rate_limit(err: Exception) -> bool:
    """Check if exception message matches common rate-limit or 429 indicator strings."""
    msg = str(err).lower()
    return any(k in msg for k in ["rate limit", "rate_limit", "429", "too many requests"])


def cooldown(seconds: int = 60) -> None:
    """Pause execution and reset event history to clear the rolling token allowance."""
    print(f"Cooling down for {seconds}s to reset the per-minute allowance...")
    time.sleep(seconds)
    BUDGET.events.clear()                        # Clear stored usage events
    print("Ready.")


# Output current usable token budget limit
print(f"Token budget: {BUDGET.limit} tokens/min (of {TPM_LIMIT} allowed)")

###**Calling Groq API Key**

This step securely fetches your API key so the AI service can be accessed, trying a few different ways depending on where the notebook is running.

* First Try: Checks if the key is already saved as an environment variable.
* Second Try: If not found, checks Google Colab's secure secrets storage.
* Last Resort: If neither works, asks you to type the key in directly (hidden as you type, like a password).
* Safety Check: Stops the notebook with a clear error if no key was ever provided, so you don't run into confusing failures later on.

In [ ]:
# Primary check: search for existing environment variable
groq_api_key = os.environ.get("GROQ_API_KEY")

if not groq_api_key:
    try:
        # Secondary check: attempt retrieval from Google Colab Secrets
        from google.colab import userdata
        groq_api_key = userdata.get("GROQ_API_KEY")
    except Exception:
        # Fallback check: prompt user for masked interactive password input
        from getpass import getpass
        groq_api_key = getpass("Enter your GROQ_API_KEY: ")

# Export API key into environment variables for global access across packages
os.environ["GROQ_API_KEY"] = groq_api_key or ""

# Enforce non-empty key validation to halt execution if credentials are missing
assert os.environ["GROQ_API_KEY"], "GROQ_API_KEY is not set."
print("Groq API key loaded.")

### **Rate-Limited LLM Wrapper & Initialization**

This step creates the AI connection itself, wrapped with the speed-limiter logic built earlier, and prepares two versions of it for different needs.

* Model Choice: Sets which AI model to use, with a lighter backup option noted in case the first one has stricter usage limits.
* Speed-Aware AI: Builds a version of the AI connection that automatically checks the usage tracker before every request, waits if needed, and retries safely if a "slow down" response comes back.
* Two Personalities:
  - **Precise Mode**: Used for tasks like database lookups, where answers need to be consistent and predictable.
  - **Creative Mode**: Used when more varied or natural-sounding phrasing is wanted.
* Ready Check: Confirms both AI connections are set up and ready to use.

In [ ]:
# Default model selection for pipeline and agent execution
MODEL_NAME = "openai/gpt-oss-120b"
# Alternative fallback model for lower token ceilings:
# MODEL_NAME = "openai/gpt-oss-20b"


class RateLimitedChatGroq(ChatGroq):
    """ChatGroq wrapper that embeds rate-limiting and retry logic inside the LLM generation step."""

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # Calculate pessimistic pre-call estimate (input tokens + response allocation cap)
        estimated = estimate_messages_tokens(messages) + (self.max_tokens or 512)

        for attempt in range(RATE_RETRIES):
            # Reserve token allocation before making the API request
            BUDGET.reserve(estimated)
            try:
                result = super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)

                # Reconcile estimated cost against actual usage returned by API response
                try:
                    usage = (result.llm_output or {}).get("token_usage", {})
                    actual = usage.get("total_tokens", 0)
                    if actual:
                        BUDGET.settle(estimated, int(actual))
                except Exception:
                    pass  # Non-critical metric collection failure

                return result

            except Exception as e:
                # Re-raise error if not a 429 rate limit or if retry attempts are exhausted
                if not is_rate_limit(e) or attempt == RATE_RETRIES - 1:
                    raise

                # Penalize rate limiter and wait out backoff period requested by server
                BUDGET.penalise()
                wait = parse_retry_after(str(e))
                print(f"  [429] rate limited - waiting {wait:.1f}s "
                      f"(attempt {attempt + 1}/{RATE_RETRIES})")
                time.sleep(wait)

        raise RuntimeError("Exhausted rate-limit retries")


# Initialize deterministic LLM instance for standard pipeline and database tasks
llm = RateLimitedChatGroq(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=LLM_MAX_TOKENS,
    max_retries=0,                  # Disable native retries to let RateLimitedChatGroq handle backoffs
    groq_api_key=groq_api_key,
)

# Initialize creative LLM instance for varied text generation
llm_high = RateLimitedChatGroq(
    model=MODEL_NAME,
    temperature=0.7,                # Higher temperature for diverse phrasing
    max_tokens=LLM_MAX_TOKENS,
    max_retries=0,
    groq_api_key=groq_api_key,
)

print("Rate-limited LLMs ready:", MODEL_NAME)

###**Safe LLM call helper**

This script makes sure a message actually gets through, even if the connection hiccups along the way.

* Retry Helper: If sending a message fails because of a temporary issue (like a timeout or the service being briefly overloaded), it waits a bit and tries again - up to a set number of attempts.
* Smart Waiting: Each retry waits a little longer than the last, so it doesn't hammer the service repeatedly in a short burst.
* Give-Up Point: If it's not a temporary issue, or all attempts are used up, it stops trying and reports the problem instead of getting stuck.
* Text Cleaner: Tidies up extra spaces and trims text down to a safe length, so what gets sent stays within limits.

In [ ]:
def safe_llm_call(messages, model=None, retries: int = 3) -> str:
    """Invoke the LLM and return its text.

    Rate limiting is handled inside RateLimitedChatGroq._generate, so this only needs to
    cover non-rate-limit transients (timeouts, 502/503, dropped connections).
    """
    model = model or llm          # default to the module-level deterministic LLM
    last_err = None
    for attempt in range(retries):
        try:
            return model.invoke(messages).content
        except Exception as e:
            last_err = e
            msg = str(e).lower()
            # note: "rate limit" is included here as a safety net, though _generate
            # should already have handled it before the exception reaches this point
            transient = any(k in msg for k in
                            ["timeout", "overloaded", "503", "502", "connection", "rate limit"])
            if not transient or attempt == retries - 1:
                break          # non-transient error, or out of retries - stop and report
            wait = 5.0 * (2 ** attempt) + random.uniform(0, 2)   # exponential backoff + jitter
            print(f"  [retry {attempt + 1}/{retries}] {type(e).__name__} - waiting {wait:.1f}s")
            time.sleep(wait)
    print(f"  [LLM error] {last_err}")
    return ""          # fail soft - caller gets an empty string instead of a crash


def clip(text: str, n: int) -> str:
    """Collapse whitespace and hard-truncate. Keeps prompts small and predictable."""
    text = re.sub(r"\s+", " ", str(text or "")).strip()   # normalize None/whitespace-only input too
    return text[:n]        # hard cap on length, not on token count - keep prompts bounded

#<font color=blue>**Building an SQL agent**

###Customer database

This step connects to the customer database and checks that it's actually loaded correctly before moving on.

* Locate the Database: Automatically points to the right file location, whether running in Google Colab or locally.
* Connect: Links the database to LangChain so the AI can query it.
* Check Tables: Lists out what tables are available inside the database.
* Safety Check: Stops the notebook right away with a clear error if the database turns out to be empty or missing, instead of letting confusing errors show up later.

In [ ]:
DB_PATH = "/content/customer.db" if os.path.isdir("/content") else "customer.db"   # Colab vs local path

db = SQLDatabase.from_uri(______)                                                  #Complete the code by providing the SQLite connection string built from DB_PATH

TABLES = db.get_usable_table_names()   # tables the LLM is allowed to see/query
print("Tables:", TABLES)
assert TABLES, "Database has no tables - check DB_PATH."   # fail fast if the DB is empty/missing

This step gives the AI a toolkit for working with the customer database.

* Toolkit Setup: Connects the database and the AI together, creating a set of ready-made tools the AI can use.
* Available Tools: These tools let the AI look at the database structure, check if a query is written correctly, and run queries to get results.
* Careful Naming: Saved as its own separate name so it doesn't accidentally get overwritten later when other tools are created.
* Verification: Prints out the names of these tools, just to confirm everything is set up and available for use.

In [ ]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)   # gives the LLM query/schema-inspection tools for `db`
sql_tools = toolkit.get_tools()          # named sql_tools (not tools) so it doesn't collide
                                         # notebook reused `tools` later and clobbered these
for t in sql_tools:
    print(t.name)   # sanity check - confirm the expected SQL tools were created

###SQL Agent

This step gives the model clear instructions and turns it into a working database assistant.

* Instructions: Tells the model exactly how to behave - fetch the requested data and reply with just the values, no extra explanation.
* Assistant Builder: Combines the AI, the database tools, and these instructions into one ready-to-use assistant, scoped only to the database tools.
* Compatibility Handling: Quietly adjusts for small differences between versions of the underlying library, so this works regardless of which version is installed.
* Ready Check: Confirms the database assistant has been created successfully.

In [ ]:
sql_system_message = "______"                                                     #Complete the code by writing a system message that tells the SQL assistant how to respond

def build_agent(model, tool_list, system_message):
    """create_react_agent renamed this argument between LangGraph versions
    (state_modifier -> prompt), so try both instead of pinning to one."""
    try:
        return create_react_agent(model, tool_list, prompt=system_message)   # newer LangGraph API
    except TypeError:
        return create_react_agent(model, tool_list, state_modifier=system_message)   # older LangGraph API

db_agent = build_agent(______, sql_tools, sql_system_message)                      #Complete the code by calling the llm
print("SQL agent ready.")

Now that the SQLite database connection is established and the SQL toolset is loaded, this step wraps those tools into a dedicated, version-compatible ReAct agent (db_agent). By constraining its system prompt to output clean, unformatted text without commentary, the agent is prepared to cleanly retrieve user profiles and preferences required by downstream search and recommendation steps.

###Database Verification

With the db_agent fully constructed, this step runs a test query against the database to extract all unique user email addresses while enforcing recursion limits (AGENT_RECURSION_LIMIT) to prevent loop runaway. Verifying clean database retrieval here ensures the pipeline can reliably lookup user profiles, unlocking customer preference extraction in the next stage.

In [ ]:
# Define test query targeting unique user identifier records
query = "Fetch all the unique email_id values"

# Invoke SQL agent using LangGraph dictionary schema while enforcing recursion limits
response = db_agent.invoke(
    {"messages": [HumanMessage(content=query)]},
    config={"recursion_limit": AGENT_RECURSION_LIMIT},   # Cap ReAct loop steps to avoid runaway API calls
)

# Extract and print the final message content containing the comma-separated output
print(response["messages"][-1].content)

#<font color=blue>**Define Tools**

##<font color=blue>**Tool-1: Expand search queries**

- Now that user information can be retrieved from the database, the pipeline needs to convert raw customer interests into focused search keywords.
- This step defines a function that calls the LLM once to transform interest keywords into targeted, time-sensitive news search queries, complete with fallback logic to prevent execution stalls.
- Packaging this function into a StructuredTool allows agents in the upcoming search stage to dynamically generate queries.

In [ ]:
# Set up input formatting rules for the tool
class ExpandSearchQueriesInput(BaseModel):
    interests: List[str] = Field(description="List of user interests to expand.")
    user_query: str = Field(description="The specific user query or topic to consider.")


def expand_search_queries(interests: List[str], user_query: str) -> str:
    """Expand user interests into time-sensitive news search queries using a single LLM call."""
    # Normalize comma-separated string inputs into list format if necessary
    if isinstance(interests, str):
        interests = [i.strip() for i in interests.split(",") if i.strip()]

    # Apply character and count limits to constrain prompt token usage
    interests = [clip(i, 60) for i in interests][:MAX_INTERESTS]
    if not interests:
        return json.dumps([])

    # System prompt directing the LLM to output a clean JSON array of search strings
    system_prompt = """______"""                                                   #Complete the code by providing clear instructions for generating time-sensitive search queries
    prompt = f"Interests: {json.dumps(interests)}\nUser focus: {user_query}"

    # Invoke LLM using the safe execution wrapper
    raw = safe_llm_call([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])

    # Extract JSON array from LLM response text using regex matching
    queries = []
    match = re.search(r"\[.*\]", raw, re.S)
    if match:
        try:
            parsed = json.loads(match.group(0))
            queries = [str(q).strip() for q in parsed if str(q).strip()]
        except json.JSONDecodeError:
            pass   # Fall through to fallback list on JSON parsing error

    # Fallback query generation to ensure downstream execution never stalls
    if not queries:
        queries = [f"{i} latest news" for i in interests]

    # Return final JSON string capped at MAX_INTERESTS limit
    return json.dumps(queries[:MAX_INTERESTS])


# Register function as a structured LangChain tool with Pydantic validation
expand_tool = StructuredTool.from_function(
    func=expand_search_queries,
    name="ExpandSearchQueries",
    description=("Expands user interests into time-sensitive news search queries. "
                 "Input: interests (list of strings) and user_query (string). "
                 "Returns a JSON array of query strings."),
    args_schema=ExpandSearchQueriesInput,   # Enforce input validation schema
)
print("expand_tool ready")

Having built the expand_search_queries tool and defined its input formatting rules, this step runs a quick verification test (smoke test) to confirm the function returns a valid JSON array of expanded search terms. Verifying this logic directly ensures search queries are properly formatted before feeding them into the live web search stage.

In [ ]:
print(expand_search_queries(["Artificial Intelligence", "Robotics"], "latest innovations"))

##<font color=blue>**Tool-2: Fetch News Results Using DuckDuckGo**

Building on the expanded search queries generated in the previous step, this stage executes live web searches to gather real-time news articles before passing them to the credibility filter.

- Live Web Search Execution: Queries DuckDuckGo News using expanded search terms to fetch dated, relevant news articles.
- Token Budget Control: Truncates snippets and formats results as clean JSON strings to satisfy LangGraph tool nodes while keeping token usage low.
- Data Normalization: Cleans snippet text, formats publication dates, and standardizes key names across API responses.
- Source Caching: Stores retrieved hits in a global dictionary (SEARCH_CACHE) to prevent downstream AI agents from hallucinating fake URLs.
- Structured Tool Packaging: Wraps the search logic into ddg_search_tool with explicit parameter validation to ensure smooth agent tool calls.

In [ ]:
# Set up input formatting rules for the search tool
class SearchInput(BaseModel):
    query: str = Field(description="A single news search query string.")


def _normalise(r: dict) -> Optional[Dict[str, str]]:
    """Clean raw search results and standardize dictionary key names."""
    url = r.get("href") or r.get("url") or r.get("link") or ""   # Extract URL across different key structures
    if not url:
        return None                                              # Skip results without a valid URL

    title = clip(r.get("title", ""), 100)
    body = clip(r.get("body") or r.get("excerpt") or "", MAX_BODY_CHARS)
    date = str(r.get("date", ""))[:10]                            # Format date as YYYY-MM-DD
    return {"title": title, "url": url, "body": body, "date": date}


def ddg_search(query: str) -> str:
    """Search DuckDuckGo News and return top results as a JSON string."""
    query = clip(query, 200)
    results: List[Dict[str, str]] = []

    try:
        with DDGS() as ddgs:
            try:
                # Primary attempt: search news endpoint for dated articles and snippets
                raw = list(ddgs.news(query, max_results=______))                    #Complete the code by providing the max_results value (hint: use MAX_RESULTS_PER_QUERY)
            except Exception:
                raw = []
            if not raw:
                # Fallback attempt: general web search if news search returns no hits
                raw = list(ddgs.text(query, max_results=MAX_RESULTS_PER_QUERY))
            for r in raw:
                item = _normalise(r)
                if item:
                    results.append(item)
    except Exception as e:
        print(f"  [search failed] {query[:50]!r}: {type(e).__name__}")
        return json.dumps([])                 # Return empty list on failure to avoid pipeline stalls

    # Store retrieved results in global cache keyed by URL to verify source authenticity
    for r in results:
        SEARCH_CACHE[r["url"]] = r

    return json.dumps(results, ensure_ascii=False)


# Global store for caching verified search hits
SEARCH_CACHE: Dict[str, Dict[str, str]] = {}

# Register search function as a LangChain structured tool with validated schema
ddg_search_tool = StructuredTool.from_function(
    func=ddg_search,
    name="DuckDuckGoSearch",
    description=("Searches DuckDuckGo News for recent articles. Input: query (a single "
                 f"string). Returns a JSON array of up to {MAX_RESULTS_PER_QUERY} results "
                 "with title, url, date and a short body."),
    args_schema=SearchInput,
)

# Run a smoke test and print preview of search results
print(ddg_search("artificial intelligence breakthrough")[:400])

##<font color=blue>**Tool-3: Filter Relevant and Trustworthy URLs Based on User Interests**

Following the live DuckDuckGo web search step, this stage processes raw search outputs to eliminate duplicate domains, social media spam, and low-credibility sources. By asking the LLM to return only array index numbers rather than full text, it filters candidate links efficiently before passing clean news sources to the summarization pipeline.

In [ ]:
def _coerce_results(search_results: Any) -> List[Dict[str, Any]]:
    """Convert raw input payloads (strings, dicts, lists) into a standardized list of dictionary records."""
    if isinstance(search_results, list):
        out = []
        for r in search_results:
            if isinstance(r, dict):
                out.append(r)
            elif isinstance(r, str):
                out.extend(_coerce_results(r))       # Recurse if list items are JSON strings
        return out

    if isinstance(search_results, dict):
        # Unwrap nested payload keys commonly returned by agents
        for key in ("results", "items", "sources", "data", "search_results"):
            if key in search_results and isinstance(search_results[key], (list, str)):
                return _coerce_results(search_results[key])
        return [search_results] if search_results.get("url") else []

    if isinstance(search_results, str):
        items: List[Dict[str, Any]] = []
        for blob in re.findall(r"\[.*?\]", search_results, re.S):      # Extract JSON array substrings
            try:
                parsed = json.loads(blob)
                items.extend(r for r in parsed if isinstance(r, dict))
            except json.JSONDecodeError:
                continue                                                # Skip malformed JSON blobs
        if items:
            return items

        # Fallback: parse raw URLs from free text and enrich from cache
        return [SEARCH_CACHE.get(u, {"title": "", "url": u, "body": ""})
                for u in re.findall(r"https?://\S+", search_results)]

    return []


def _enrich(record: Dict[str, Any]) -> Dict[str, str]:
    """Restore cached title and snippet body text for a retrieved URL."""
    url = str(record.get("url", "")).strip().rstrip(".,)\"'")         # Trim punctuation artifacts
    cached = SEARCH_CACHE.get(url, {})
    return {
        "title": clip(record.get("title") or cached.get("title", ""), 90),
        "url":   url,
        "body":  clip(record.get("body") or cached.get("body", ""), MAX_BODY_CHARS),
    }


# Define input formatting rules for the credibility filter tool
class CredibilityInput(BaseModel):
    results: Any = Field(description="The JSON string returned by DuckDuckGoSearch, "
                                     "or a list of {title, url, body} objects.")


def filter_with_llm(results: Any) -> str:
    """Filter candidate news sources down to reliable, non-duplicate websites."""
    items = [_enrich(r) for r in _coerce_results(results) if r.get("url")]   # Normalize records
    if not items:
        return json.dumps([])

    # Deduplicate candidate links by domain name before calling the LLM
    seen, deduped = set(), []
    for r in items:
        domain = re.sub(r"^https?://(www\.)?", "", r["url"]).split("/")[0].lower()
        if domain and domain not in seen:
            seen.add(domain)
            deduped.append(r)

    deduped = deduped[:MAX_RESULTS_TO_FILTER]                          # Cap candidate count

    # Pass a numbered list to the LLM and request index numbers back to save output tokens
    numbered = "\n".join(
        f"{i}. {r['title']} | {r['url']}" for i, r in enumerate(deduped)
    )
    system_prompt = """__________"""                                   #Complete the code by providing clear instructions for evaluating source credibility

    raw = safe_llm_call([SystemMessage(content=system_prompt),
                         HumanMessage(content=numbered)])

    kept = []
    match = re.search(r"\[[^\]]*\]", raw, re.S)
    if match:
        try:
            for n in json.loads(match.group(0)):
                if isinstance(n, int) and 0 <= n < len(deduped):          # Ignore invalid indices
                    kept.append(deduped[n])
        except (json.JSONDecodeError, TypeError):
            pass

    # Fallback to default top candidates if parsing fails or no indices are returned
    if not kept:
        kept = deduped[:MAX_URLS_TO_SUMMARIZE]

    return json.dumps(kept[:MAX_URLS_TO_SUMMARIZE], ensure_ascii=False)


# Register credibility filter function as a LangChain structured tool
credibility_tool = StructuredTool.from_function(
    func=filter_with_llm,
    name="CredibilityFilter",
    description=("Filters news search results down to credible, non-duplicate sources. "
                 "Input: results - the JSON string returned by DuckDuckGoSearch "
                 "(pass the combined results of all searches). "
                 "Returns a JSON array of credible sources with title, url and body."),
    args_schema=CredibilityInput,
)
print("credibility_tool ready")

##<font color=blue>**Tool-4: Generate summary for the URLs**

Building on the credible, filtered sources identified in the previous stage, this step synthesizes those articles into a concise, cited news briefing paired with verified source links.
- Clean Source Formatting: Formats incoming source data into structured line items rather than raw text strings, giving the LLM clear context for each entry.
- Strict Agent Guardrails: Directs the model never to ask follow-up questions or request missing details, maintaining fully autonomous execution.
- Fallback Summarization: Automatically shifts to describing general outlet coverage if real-time text snippets are missing, avoiding fabricated headlines or model refusals.
- Token Budget Control: Enforces strict limits on the maximum number of summarized URLs (MAX_URLS_TO_SUMMARIZE) to prevent token window overflow.
- Citation Verification: Appends raw source URLs directly to the generated summary for end-user verification and transparency.

In [ ]:
# Define input formatting rules for the summarizer tool
class SummarizeInput(BaseModel):
    sources: Any = Field(description="The JSON string returned by CredibilityFilter, "
                                     "or a list of {title, url, body} objects.")


def summarize_news(sources: Any) -> str:
    """Summarize credible news sources into a cited briefing string with source links."""
    # Coerce input and limit candidate sources to cap prompt size
    items = [_enrich(r) for r in _coerce_results(sources) if r.get("url")]
    items = items[:MAX_URLS_TO_SUMMARIZE]

    if not items:
        return "No credible sources were available to summarize."

    urls = [r["url"] for r in items]
    lines = [f"{i}. TITLE: {r['title'] or '(none)'}\n   URL: {r['url']}\n   SNIPPET: {r['body'] or '(none)'}"
             for i, r in enumerate(items, 1)]
    have_content = any(r["title"] or r["body"] for r in items)   # Check if valid text snippets exist

    # Enforce strict summarization rules in system prompt
    system_prompt = """___________"""                                  #Complete the code by providing clear instructions for writing the news briefing
    if not have_content:
        system_prompt += ("\nNone of these sources came with snippets, so write a short "
                          "orientation note on what each outlet covers instead.")

    prompt = "_________" + "\n".join(lines)                            #Complete the code by writing a prompt to summarize the sources

    # Call LLM to generate news summary
    summary = safe_llm_call([SystemMessage(content=system_prompt),
                             HumanMessage(content=prompt)])

    # Guard against LLM asking questions or refusing to answer by using a fallback
    if not summary or re.search(r"(could you|can you|please provide|I can't create|I cannot create)",
                                summary[:300], re.I):
        summary = "Automatic summary unavailable. Retrieved headlines:\n" + "\n".join(
            f"- {r['title'] or r['url']}: {r['body']}" for r in items)

    # Append raw source links for verification
    return summary + "\n\nSources:\n" + "\n".join(f"- {u}" for u in urls)


# Register summarization function as a LangChain structured tool
summarize_tool = StructuredTool.from_function(
    func=summarize_news,
    name="SummarizeNews",
    description=("Generates a news briefing from credible sources. "
                 "Input: sources - the JSON string returned by CredibilityFilter. "
                 "Returns the summary text plus the source links."),
    args_schema=SummarizeInput,
)
print("summarize_tool ready")

#<font color=blue>**Creating an Agent**

Having defined all four individual pipeline tools-Query Expansion, Web Search, Credibility Filtering, and Summarization-this step binds them together into a complete ReAct news research agent (agent). The system prompt enforces a strict, single-pass 5-step sequence to guide the model through end-to-end execution without repeating steps or getting stuck in loops.

In [ ]:
# Group individual tools into a unified tool list for agent binding
news_tools = [____, ____, ____, ____]                                   #Complete the code by calling the four already-defined tools

# System prompt defining strict step-by-step tool invocation rules
general_agent_system_message = """You are a news research assistant with four tools:
ExpandSearchQueries, DuckDuckGoSearch, CredibilityFilter and SummarizeNews.

Follow this order exactly and do not repeat a step:
1. ExpandSearchQueries once, with the user's interests and query.
2. DuckDuckGoSearch once per expanded query (at most 3 searches in total).
3. CredibilityFilter once. Pass the search output through unchanged as `results`.
4. SummarizeNews once. Pass the CredibilityFilter output through unchanged as `sources`.
5. Return the summary from step 4 as your final answer, with the source links.

Critical rules:
- Never retype, edit or invent a URL. Copy tool output verbatim into the next tool.
- Never ask the user a question. Produce the best answer you can from the tool output.
- Keep your own reasoning text short.
"""

# Instantiate the version-safe ReAct news agent
agent = build_agent(llm, news_tools, general_agent_system_message)
print("News agent ready.")

# Note: Tracing agent execution can be enabled via streaming or by setting os.environ["LANGCHAIN_VERBOSE"] = "true"

##<font color=blue>**Fetch user interests**

With the core tools and agents fully initialized, this step retrieves target user interest profile data from the SQLite database. It attempts to query preferences using the SQL agent first, falling back seamlessly to a direct SQLite query if the agent encounters errors or returns empty outputs, ensuring downstream news search steps always receive clean user topics.

In [ ]:
TABLE_NAME = "customers"
EMAIL_COL = "email_id"
INTEREST_COL = "interests"

In [ ]:
def parse_interests(text: str) -> List[str]:
    """Turn whatever the agent returned into a clean list of interest strings."""
    if not text:
        return []
    match = re.search(r"\[.*\]", text, re.S)          # try the JSON-array case first
    if match:
        try:
            parsed = json.loads(match.group(0))
            if isinstance(parsed, list):
                return [str(i).strip().strip('"\'') for i in parsed if str(i).strip()]
        except json.JSONDecodeError:
            pass                                        # not valid JSON - fall through to plain-text parsing
    text = re.sub(r"^[^:]{0,60}:", "", text.strip(), count=1)     # drop "The interests are:"
    parts = re.split(r"[,;\n]|\s\|\s", text.strip(" []"))          # split on common delimiters the LLM might use
    return [p.strip(" -*\"'") for p in parts if 1 < len(p.strip(" -*\"'")) < 60]   # drop empties/junk-length tokens


def fetch_interests_direct(email: str) -> List[str]:
    """Deterministic fallback - no LLM, no tokens."""
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(
            f"SELECT {INTEREST_COL} FROM {TABLE_NAME} WHERE {EMAIL_COL} = ?", (email,)
        ).fetchall()
    out: List[str] = []
    for (val,) in rows:
        out.extend(i.strip() for i in str(val).split(",") if i.strip())   # flatten in case of multiple/CSV rows
    return out


def fetch_interests(email: str, use_agent: bool = True) -> List[str]:
    interests: List[str] = []
    if use_agent:
        try:
            result = db_agent.invoke(
                {"messages": [HumanMessage(content=(
                    f"Return the {INTEREST_COL} for {EMAIL_COL} = '{email}' from the "
                    f"{TABLE_NAME} table as a comma-separated list. No commentary."
                ))]},
                config={"recursion_limit": AGENT_RECURSION_LIMIT},
            )
            interests = parse_interests(result["messages"][-1].content)
        except Exception as e:
            print(f"  [SQL agent failed: {type(e).__name__}] falling back to direct SQL")   # agent errored - don't fail the whole call

    if not interests:
        interests = fetch_interests_direct(email)          # covers both agent failure and agent returning nothing usable

    # Deduplicate, preserve order, cap.
    seen, cleaned = set(), []
    for i in interests:
        k = i.lower()
        if k not in seen:
            seen.add(k)
            cleaned.append(i)
    return cleaned[:MAX_INTERESTS]


print(fetch_interests("emma.a88fec03-c@gmail.com"))

##<font color=blue>**Retrieve URLs and Summaries for Three Areas of Interest**

With user preference retrieval established and all four research tools integrated into the agent, this final stage ties the entire system together. It provides both a dynamic agentic execution path and a fast, direct pipeline option (run_pipeline_directly) that saves roughly 80% of token costs, backed by automatic rate-limit recovery and fallbacks.

In [ ]:
def run_pipeline_directly(interests: List[str], user_query: str) -> str:
    """Run tools in a fixed, step-by-step order without AI agent reasoning overhead."""
    # Step 1: Expand user interests into specific search keywords
    queries = json.loads(expand_search_queries(interests, user_query))
    print("Queries:", queries)

    # Step 2: Search DuckDuckGo once for each generated query keyword and pool results
    combined = []
    for q in queries:
        combined.extend(json.loads(ddg_search(q)))
    print(f"Collected {len(combined)} raw results")

    # Step 3: Filter raw search results down to credible, unique news sources
    filtered = filter_with_llm(combined)
    print(f"Kept {len(json.loads(filtered))} credible sources")

    # Step 4: Summarize filtered news articles into a cited news briefing
    return summarize_news(filtered)


def query_response(email: str, user_query: str, use_agent: Optional[bool] = None) -> str:
    """Execute complete workflow: Email -> Interests -> Queries -> Search -> Filter -> Summary."""
    # Determine whether to use agentic loop or direct pipeline based on global setting/override
    use_agent = USE_AGENT if use_agent is None else use_agent

    # Retrieve user interest profile from database
    interests = fetch_interests(email, use_agent=use_agent)
    if not interests:
        msg = f"No interests found for {email}. Check the email address against the database."
        print(msg)
        return msg
    print(f"Interests for {email}: {interests}")

    # Path A: Run direct deterministic pipeline without agent overhead
    if not use_agent:
        final = run_pipeline_directly(interests, user_query)
    # Path B: Run interactive ReAct AI agent with fallback protection
    else:
        agent_prompt = (
            f"User interests: {json.dumps(interests)}\n"
            f"User is specifically looking for: {user_query}\n"
            "Run the four-step process from your instructions and return the final summary."
        )
        final = ""
        try:
            # Invoke agent with recursion safety limit
            result = agent.invoke(
                {"messages": [HumanMessage(content=agent_prompt)]},
                config={"recursion_limit": AGENT_RECURSION_LIMIT},
            )
            final = result["messages"][-1].content
            if not final.strip():
                raise ValueError("empty agent response")

        except Exception as e:
            # Handle rate limit 429 errors by waiting out the rate limit window and retrying once
            if is_rate_limit(e):
                wait = parse_retry_after(str(e), default=60.0)
                print(f"  [agent hit the TPM ceiling] waiting {wait:.0f}s and retrying once")
                time.sleep(wait)
                BUDGET.events.clear()           # Reset rate limit tracker after waiting
                try:
                    result = agent.invoke(
                        {"messages": [HumanMessage(content=agent_prompt)]},
                        config={"recursion_limit": AGENT_RECURSION_LIMIT},
                    )
                    final = result["messages"][-1].content
                except Exception as e2:
                    print(f"  [agent failed again: {type(e2).__name__}] using the direct pipeline")
            else:
                print(f"  [agent path failed: {type(e).__name__}: {e}] using the direct pipeline")

        # Fallback to direct pipeline path if agent fails or outputs empty content
        if not final.strip():
            final = run_pipeline_directly(interests, user_query)

    print("\n======= FINAL RESPONSE =======")
    print(final)
    return final

#<font color=blue>**Test cases**

## Test Case 1

In [ ]:
# Check for the user ID
email = "______"                                                                 #Complete the code by providing a user email
user_query = "______"                                                            #Complete the code by providing the relevant user query
response = query_response(email, user_query)   # runs the full pipeline end-to-end and prints the final summary

## Test Case 2

In [ ]:
# Check for the user ID
#cooldown(60)   # let the per-minute allowance refill before the next user
email = "______"                                                                 #Complete the code by providing a user email
user_query = "______"                                                            #Complete the code by providing the relevant user query
response = query_response(email, user_query)   # runs the full pipeline end-to-end and prints the final summary

## Test Case 3

In [ ]:
#Check for the user ID
#cooldown(60)   # let the per-minute allowance refill before the next user
email = "______"                                                                 #Complete the code by providing a user email
user_query = "______"                                                            #Complete the code by providing the relevant user query
response = query_response(email, user_query)

##<font color=blue>**Conclusion**

-